# Test word search in the documents

In [9]:
import pandas as pd

df = pd.read_csv('output/2025-10-31/site_data.csv')
# deduplicate by 'Agency','Facility_Name'
df_dedup = df.drop_duplicates(subset=['Agency', 'Facility_Name'])
print(df_dedup.shape)
# print the % of 'Shared_PDF'==Yes
shared_pdf_percent = (df_dedup['Shared_PDF'] == 'Yes').mean() * 100
print(f"Percentage of Shared_PDF == 'Yes': {shared_pdf_percent:.2f}%")
# print % of each 'region' 
region_percent = df_dedup['Region'].value_counts(normalize=True) * 100
print("Percentage of each region:")
print(region_percent.sort_index())
# print % of each 'Major/Minor'
major_minor_percent = df_dedup['Major/Minor'].value_counts(normalize=True) * 100
print("Percentage of each Major/Minor:")
print(major_minor_percent.sort_index())

(142, 7)
Percentage of Shared_PDF == 'Yes': 13.38%
Percentage of each region:
Region
1     13.380282
2     17.605634
3      7.746479
4      6.338028
5F     2.112676
5R     9.859155
5S    19.718310
7      6.338028
8      6.338028
9     10.563380
Name: proportion, dtype: float64
Percentage of each Major/Minor:
Major/Minor
Major           64.788732
Major, Minor     8.450704
Minor           26.760563
Name: proportion, dtype: float64


In [19]:
n_train_data = 15
print('For ' + str(n_train_data) + ' facilities as train examples we need :')
# int value closest
print((region_percent.sort_index() / 100 * n_train_data).round())
print((major_minor_percent.sort_index() / 100 * n_train_data).round())

For 15 facilities as train examples we need :
Region
1     2.0
2     3.0
3     1.0
4     1.0
5F    0.0
5R    1.0
5S    3.0
7     1.0
8     1.0
9     2.0
Name: proportion, dtype: float64
Major/Minor
Major           10.0
Major, Minor     1.0
Minor            4.0
Name: proportion, dtype: float64


In [ ]:
# Find n_train_data rows such that the ratio of regions and major/minor in the sample matches the overall distribution as closely as possible.

import numpy as np

# Method: Stratified sampling by Region with exact target counts
try:
    # Calculate target counts per region
    region_counts = (region_percent / 100 * n_train_data).round().astype(int)
    
    # Adjust to ensure we get exactly n_train_data samples
    diff = n_train_data - region_counts.sum()
    if diff > 0:
        # Add to the largest regions
        for region in region_counts.nlargest(diff).index:
            region_counts[region] += 1
    elif diff < 0:
        # Remove from the largest regions
        for region in region_counts.nlargest(-diff).index:
            region_counts[region] = max(0, region_counts[region] - 1)
    
    print("Target counts per region:")
    print(region_counts.sort_index())
    
    # Sample from each region according to target counts
    train_samples = []
    for region, count in region_counts.items():
        if count > 0:
            region_data = df_dedup[df_dedup['Region'] == region]
            if len(region_data) >= count:
                sample = region_data.sample(n=count, random_state=42)
            else:
                # If not enough samples, take all available
                sample = region_data
                print(f"Warning: Only {len(region_data)} samples available for Region {region}, but {count} requested")
            train_samples.append(sample)
    
    train_sample = pd.concat(train_samples, ignore_index=False)
    
    print(f"\nSelected {len(train_sample)} training samples")
    print("\nActual distribution in sample:")
    print("Region distribution:")
    print(train_sample['Region'].value_counts().sort_index())
    print("\nMajor/Minor distribution:")
    print(train_sample['Major/Minor'].value_counts().sort_index())
    
    print("\nComparison with expected distribution:")
    print("Region (Expected vs Actual):")
    expected_region = (region_percent / 100 * n_train_data).round()
    actual_region = train_sample['Region'].value_counts().sort_index()
    # Reindex to show all expected regions
    all_regions = expected_region.index.union(actual_region.index)
    comparison_region = pd.DataFrame({
        'Expected': expected_region.reindex(all_regions, fill_value=0),
        'Actual': actual_region.reindex(all_regions, fill_value=0)
    })
    print(comparison_region)
    
    print("\nMajor/Minor (Expected vs Actual):")
    expected_major = (major_minor_percent / 100 * n_train_data).round()
    actual_major = train_sample['Major/Minor'].value_counts().sort_index()
    all_major = expected_major.index.union(actual_major.index)
    comparison_major = pd.DataFrame({
        'Expected': expected_major.reindex(all_major, fill_value=0),
        'Actual': actual_major.reindex(all_major, fill_value=0)
    })
    print(comparison_major)
    
except Exception as e:
    print(f"Error with stratified sampling: {e}")  
    import traceback
    traceback.print_exc()
    print("Falling back to simple random sample")
    train_sample = df_dedup.sample(n_train_data, random_state=42)

# Display the selected facilities
print("\nSelected facilities:")
print(train_sample[['Agency', 'Facility_Name', 'NPDES_No', 'PDF_File','Region', 'Major/Minor','Shared_PDF']].head(20))


Selected 15 training samples

Actual distribution in sample:
Region distribution:
Region
1     3
2     3
3     1
4     2
5F    1
5R    2
5S    1
8     2
Name: count, dtype: int64

Major/Minor distribution:
Major/Minor
Major           7
Major, Minor    2
Minor           6
Name: count, dtype: int64

Comparison with expected distribution:
Region (Expected vs Actual):
        Expected  Actual
Region                  
1            2.0     3.0
2            3.0     3.0
3            1.0     1.0
4            1.0     2.0
5F           0.0     1.0
5R           1.0     2.0
5S           3.0     1.0
7            1.0     NaN
8            1.0     2.0
9            2.0     NaN

Major/Minor (Expected vs Actual):
              Expected  Actual
Major/Minor                   
Major             10.0       7
Major, Minor       1.0       2
Minor              4.0       6

Selected facilities:
                                                Agency  \
42                                  Heritage Ranch CSD   
58   

/tmp/ipykernel_82771/153612594.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dedup['strata'] = df_dedup['Region'].astype(str) + '_' + df_dedup['Major/Minor'].astype(str)
/tmp/ipykernel_82771/153612594.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_sample = df_dedup.groupby('strata', group_keys=False).apply(
